## Formulario correo

In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)



In [7]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')

# df_target_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
# df_target_desembolso.rename(columns={'FECHA_DESEMBOLSOS': 'fecha_desembolso'}, inplace=True)
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['target'] = 1

filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','MONTO','ASESOR','CANALVENTA']].copy()

df_desembolso = df_target_desembolso[['DNI']].drop_duplicates().merge(
    df_fugas[['DNI']].drop_duplicates(),
    on=['DNI'],
    how='inner'
)

df_desembolso = df_target_desembolso.drop_duplicates().merge(
    df_fugas.drop_duplicates(),
    on=['DNI'],
    how='inner'
)
df_desembolso['target'] = df_desembolso['target'].fillna(0).astype(int)
df_desembolso['fugas'] = (df_desembolso['target'] == 0).astype(int)
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)

df_desembolso['dni_cliente'] = (
    df_desembolso['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)



In [8]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)

df_formato['fecha_visita']='2026-07-29'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'
df_formato.shape

(1801, 19)

In [9]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_formato['retiro'] = (
    df_formato['dni_cliente'].isin(dni_retiro) |
    df_formato['celular'].isin(cel_retiro)
).astype(int)

C:\Users\DATA\AppData\Local\Temp\ipykernel_12856\1824424006.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [10]:
df_seguimiento_1 = df_formato.merge(
    df_desembolso,
    on=['dni_cliente'],
    how='left'
)

df_seguimiento_1['target'] = df_seguimiento_1['target'].fillna(0).astype(int)
df_seguimiento_1['fugas'] = df_seguimiento_1['fugas'].fillna(0).astype(int)

In [11]:
filename='fomato_agendas_alfin_credicash_2026.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_lista = pd.read_csv(ruta_archivo,sep=';')
print(df_lista.columns.tolist())

['dni_cliente', 'nombre_cliente', 'celular', 'cod_agencia', 'agencia_atencion', 'fecha_visita', 'monto']


In [12]:
df_lista['dni_cliente'] = (
    df_lista['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)

In [13]:
query = f"""
	SELECT * FROM Alice.prospectos_envio_alfin 
    where estado='procesado'
	and DATE(fecha_envio)>='2026-07-01'
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='ENVIADO'
	and DATE(fecha_envio)>='2026-07-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [14]:
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db2.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar=df_validar_01.unionByName(df_validar_02)

filename='hola.csv'
aja=cargar_archivo_csv(spark,filename,';',True)
aja = aja.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
)
aja=aja.dropDuplicates(['dni_cliente'])
# overwrite_table_SQL(spark,aja,f'subir_borrar',server_kishin,user_kishin,pwd_kishin,'DANTALION')
# overwrite_table_SQL(spark,aja,f'ALFIN_BORRAR_BORRAR_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')

NameError: name 'spark' is not defined

In [ ]:
df_validar=df_validar.withColumnRenamed('DNI','dni_cliente')

In [ ]:
aja=aja.join(df_validar.select('dni_cliente','COLOR_FINAL'),['dni_cliente'],'left')

+-----------+-----------+-----------+------------------+-----------+---------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+----------+---------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+------------------+---------+
|dni_cliente|COLOR_FINAL|COD_USER_V3|           USER_V3|  PERFIL_RO|  campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|GRUPO_TASA|TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA_1_SS|TASA_2_SS|TASA_3_SS|TASA_4_SS|TASA_5_SS|TASA_6_SS|TASA_7_SS|ALERTA_MAQUETA| FEN|PERFIL_ESPECIAL|TASA_MIN_DESCUENTO|     TIPO|
+-----------+-----------+-----------+-----------

In [15]:
df_prospectos_correos_alfin['dia_ref'] = pd.to_datetime(
    df_prospectos_correos_alfin['fecha_envio']
).dt.date

df_prospectos_envio_alfin['dia_ref'] = pd.to_datetime(
    df_prospectos_envio_alfin['fecha_envio']
).dt.date

df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente','dia_ref'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente','dia_ref'])

df_seguimiento = df_prospectos_envio_alfin[['dni_cliente', 'dia_ref']].merge(
    df_prospectos_correos_alfin[['dni_cliente', 'dia_ref','celular']],
    on=['dni_cliente', 'dia_ref'],
    how='inner'
)

In [16]:
df_seguimiento_1 = df_seguimiento.merge(
    df_desembolso,
    on=['dni_cliente'],
    how='left'
)

df_seguimiento_1['target'] = df_seguimiento_1['target'].fillna(0).astype(int)
df_seguimiento_1['fugas'] = df_seguimiento_1['fugas'].fillna(0).astype(int)

In [17]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

filename='desembolso.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
df6 = df_des.copy()
df6["celular"] = None
df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()



# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4,df6],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_seguimiento_1['retiro'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_retiro) |
    df_seguimiento_1['celular'].isin(cel_retiro)
).astype(int)

C:\Users\DATA\AppData\Local\Temp\ipykernel_12856\3506000229.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [18]:
df_lista = df_lista[
    ~(
        df_lista['dni_cliente'].isin(dni_retiro) |
        df_lista['celular'].isin(cel_retiro)
    )
].copy()

In [19]:
dni_retiro = set(df_seguimiento_1['dni_cliente'].dropna())
cel_retiro = set(df_seguimiento_1['celular'].dropna())



In [20]:
df_lista_1=aja.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'hola.csv')
df_lista_1.to_csv(ruta_archivo, index=False,sep=';')

NameError: name 'aja' is not defined

In [21]:
df_lista
df_prospectos_envio_alfin

,id,hash_duplicado,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion,estado,codigo_http_ms,respuesta_ms,fecha_creacion,fecha_envio,dia_ref
0,3,3a6b361a91aa763c094d1f5c7f164cc2,00000001,TARGET,19337433,JUANA ESTHER VERA MURILLO DE ZAVALETA,963517955,734265 - TRUJ CENTRO,2026-07-04,4000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-03 02:06:01,2026-07-03 10:57:08,2026-07-03
1,4,a3faa0a4c0f17ed48834205c8f9db5d0,00000001,TARGET,19998396,ANGEL LUIS PONCE HUARINGA,937474638,734280 - PC HUANCAYO,2026-07-04,14000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-03 02:06:01,2026-07-03 10:57:10,2026-07-03
2,5,845267229900e4e2b150f46a62b9e697,00000001,TARGET,40949266,ADA ESMERALDA AYALA HERRERA,943587132,736568 - AREQ PAMPILLA,2026-07-04,3000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-03 02:06:02,2026-07-03 10:57:13,2026-07-03
3,6,1255da78d65eca60f06ee6056f033b32,00000001,TARGET,40319405,ROBERTO SEGUNDO ZEVALLOS CERPA,923809089,736568 - AREQ PAMPILLA,2026-07-04,9900,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-03 02:06:02,2026-07-03 10:57:16,2026-07-03
4,7,64d55d358e0eb1ac17fe2dcef24dc99f,00000001,TARGET,00809150,ROGER VALLES RODRIGUEZ,987302232,733824 - TARAPOTO,2026-07-04,9800,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-03 02:06:03,2026-07-03 10:57:18,2026-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37578,217629,None,BOT,TARGET,80522340,NORA ISABEL TORRES CENTURION,978379293,734281 - CHICLAYO BALTA,2026-07-24,19600,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-28 12:35:27,2026-07-28 13:48:21,2026-07-28
37579,217630,None,BOT,TARGET,80590461,ROSALIA ANTONIA QUILLUYA TAYPE,951718085,738381 - ENMANCIPACION,2026-07-24,3600,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-28 12:35:27,2026-07-28 13:50:27,2026-07-28
37580,217631,None,BOT,TARGET,80598414,MARIA KARINA VALENCIA COLAN,937763998,739470 - HUARAL,2026-07-24,20000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-28 12:35:27,2026-07-28 13:54:05,2026-07-28
37581,217632,None,BOT,TARGET,80599871,ROMEO RUBEN ROSALES MAYA,910051873,739470 - HUARAL,2026-07-24,11600,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-07-28 12:35:27,2026-07-28 13:56:11,2026-07-28


### spark -- datos faltantes

In [9]:
df_seguimiento_1.head()

,dni_cliente,nombre_cliente,celular,cod_agencia,agencia_atencion,fecha_visita,monto_solicitado,supervisor,canal_campo,codigo_ejecutivo_id,...,dni_vendedor,agencia_tienda,operador,tipo_gestion,retiro,target,MONTO,ASESOR,CANALVENTA,fugas
0,22435762,GEORGINA OLGA AQUINO CABELLO,920239053,735996 - HUANUCO,HUANUCO,2026-07-25,29200,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,...,BOT,735996 - HUANUCO,TARGET,Derivacion,1,0,NaN,NaN,NaN,0
1,41523629,ROXANA NOA SOTO,982409792,737870 - VILLA MARIA 2,VILLA MARIA 2,2026-07-25,29200,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,...,BOT,737870 - VILLA MARIA 2,TARGET,Derivacion,1,0,NaN,NaN,NaN,0
2,16734054,maritzza nelly valdera leon,968850999,734281 - CHICLAYO BALTA,CHICLAYO BALTA,2026-07-25,29200,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,...,BOT,734281 - CHICLAYO BALTA,TARGET,Derivacion,1,0,NaN,NaN,NaN,0
3,41502255,SEGUNDO LIZARDO RODRIGUEZ RODRIGUEZ,996528058,737896 - SAN JUAN DE LURIG,SAN JUAN DE LURIG,2026-07-25,29200,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,...,BOT,737896 - SAN JUAN DE LURIG,TARGET,Derivacion,0,0,NaN,NaN,NaN,0
4,22246340,POLINARIO NAVARRO PERALTA,956472464,734264 - PISCO,PISCO,2026-07-25,29200,CARLOS ENRIQUE RAMIREZ CACHIQUE,CALL CENTER / TARGET OUTSOURCING,00000001,...,BOT,734264 - PISCO,TARGET,Derivacion,1,0,NaN,NaN,NaN,0


In [11]:
ruta_archivo = os.path.join(ruta_csv, 'muestra_alfin_correo_formulario_tmp.csv')
df_seguimiento_1.to_csv(ruta_archivo, index=False,sep=';')
# ruta_archivo = os.path.join(ruta_alfin, 'desembolso.csv')
# df_des.to_csv(ruta_archivo, index=False,sep=';')

In [34]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [13]:
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db2.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar=df_validar_01.unionByName(df_validar_02)

filename='muestra_alfin_correo_formulario_tmp.csv'
df_tmp_dni=cargar_archivo_csv(spark,filename,';',True)

# filename='retiro_alfin_acum.csv'
# df_tmp_retiro=cargar_archivo_csv(spark,filename,';',True)

In [ ]:

filename='muestra_alfin_correo_formulario_tmp.csv'
aja=cargar_archivo_csv(spark,filename,';',True)
aja = aja.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
)
aja=aja.dropDuplicates(['dni_cliente'])
overwrite_table_SQL(spark,aja,f'subir_borrar',server_kishin,user_kishin,pwd_kishin,'DANTALION')
# overwrite_table_SQL(spark,aja,f'ALFIN_BORRAR_BORRAR_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')

In [21]:
df_validar=df_validar.withColumnRenamed('DNI','dni_cliente')

In [27]:
df_formato_alfin=aja.join(df_validar,['dni_cliente'],'left')

In [28]:
df_formato_alfin = df_formato_alfin.withColumn(
    "RANGO_OFERTA",   
    F.when(F.col("OFERTA_MAX").cast("int").isNull(), "SIN DATO")
     .when(F.col("OFERTA_MAX").cast("int") < 2500,  "00.[0 - 2,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 5000,  "01.[2,500 - 5,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 7500,  "02.[5,000 - 7,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 10000, "03.[7,500 - 10,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 12500, "04.[10,000 - 12,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 15000, "05.[12,500 - 15,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 17500, "06.[15,000 - 17,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 20000, "07.[17,500 - 20,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 22500, "08.[20,000 - 22,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 25000, "09.[22,500 - 25,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 27500, "10.[25,000 - 27,500)")
     .otherwise("11.[27,500 A MÁS]")
)



In [35]:
# df_formato_alfin.filter(F.col('OFERTA_MAX')>=10000).count()
df_formato_alfin=df_formato_alfin.filter(F.col('OFERTA_MAX')>=10000)

In [36]:
df_pivot = (
    df_formato_alfin
    .groupBy("RANGO_OFERTA")
    .pivot("PROPENSION_DISTRIBUCION")
    .agg(F.count("dni_cliente"))
    .fillna(0)
    .orderBy("RANGO_OFERTA")
)

df_pivot.show(30, truncate=False)

+--------------------+---+---+---+---+---+---+
|RANGO_OFERTA        |1  |2  |3  |4  |5  |6  |
+--------------------+---+---+---+---+---+---+
|04.[10,000 - 12,500)|461|234|21 |19 |13 |15 |
|05.[12,500 - 15,000)|452|220|17 |20 |8  |3  |
|06.[15,000 - 17,500)|343|187|20 |10 |8  |8  |
|07.[17,500 - 20,000)|373|275|20 |11 |13 |11 |
|08.[20,000 - 22,500)|432|265|40 |15 |6  |4  |
|09.[22,500 - 25,000)|181|153|17 |6  |4  |5  |
|10.[25,000 - 27,500)|151|127|23 |9  |3  |2  |
|11.[27,500 A MÁS]   |7  |5  |0  |0  |0  |0  |
+--------------------+---+---+---+---+---+---+



In [ ]:
print(df_validar.columns)
print(df_validar.columns)

['dni_cliente', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO']
['dni_cliente', 'nombre_cliente', 'celular', 'cod_agencia', 'agencia_atencion', 'fecha_visita', 'monto_solicitado', 'supervisor', 'canal_campo', 'codigo_ejecutivo_id', 'ejecutivo_target', 'cdv_alfin_banco', 'hora_visita', 'telefono_cliente', 'dni_vendedor', 'agencia_tienda', 'operador', 'tipo_gestion', 'retiro', 'target', 'MONTO', 'ASESOR', 'CANALVENTA', 'fugas']


In [47]:
from pyspark.sql.window import Window

w = Window.partitionBy('PROPENSION_DISTRIBUCION','FRESCURA',"USER_V3").orderBy(F.rand())

df_split = df_formato_alfin.withColumn("grupo_split", F.ntile(5).over(w))

df_parte_1 = df_split.filter(F.col("grupo_split").isin(2,4,5)).drop("grupo_split")
# df_parte_2 = df_split.filter(F.col("grupo_split") == 2).drop("grupo_split")
# df_parte_3 = df_split.filter(F.col("grupo_split") == 3).drop("grupo_split")
# df_parte_4 = df_split.filter(F.col("grupo_split") == 4).drop("grupo_split")
# df_parte_5 = df_split.filter(F.col("grupo_split") == 5).drop("grupo_split")
# df_parte_6 = df_split.filter(F.col("grupo_split") == 6).drop("grupo_split")
# df_parte_7 = df_split.filter(F.col("grupo_split") == 7).drop("grupo_split")

print(
    df_parte_1.count()
    # df_parte_2.count(),
    # df_parte_3.count(),
    # df_parte_4.count(),
    # df_parte_5.count(),
    # # df_parte_6.count(),
    # df_parte_7.count()
)

2430


In [ ]:
['dni_cliente', 'nombre_cliente', 'celular', 'cod_agencia', 'agencia_atencion', 'fecha_visita', 'monto_solicitado', 'supervisor', 'canal_campo', 'codigo_ejecutivo_id', 'ejecutivo_target', 'cdv_alfin_banco', 'hora_visita', 'telefono_cliente', 'dni_vendedor', 'agencia_tienda', 'operador', 'tipo_gestion', 'retiro', 'target', 'MONTO', 'ASESOR', 'CANALVENTA', 'fugas', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TASA_MIN_DESCUENTO', 'TIPO', 'RANGO_OFERTA']ofe

SyntaxError: invalid syntax (121387668.py, line 1)

In [50]:
df_parte_1=df_parte_1.select('dni_cliente','nombre_cliente','celular','cod_agencia', 'agencia_atencion','COLOR_FINAL','OFERTA_MAX')

In [51]:
df_envio=df_parte_1.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'hola.xlsx')
df_envio.to_excel(ruta_archivo, index=False)

In [45]:
print(790+878+762)

2430


### completar base + columnas de consulta campaña

In [13]:
dni_en_base = set(df_ref_base_pd['dni_cliente'].dropna())

df_seguimiento_1['en_base'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_en_base) 
).astype(int)

In [14]:
df = df_seguimiento_1.copy()

# Asegurar que dia_ref sea fecha
df['dia_ref'] = pd.to_datetime(df['dia_ref'], errors='coerce')

# Cantidad de envíos por DNI
df['q_envios'] = df.groupby('dni_cliente')['dni_cliente'].transform('size')

df_resumen = (
    df.sort_values('dia_ref')
      .drop_duplicates(subset='dni_cliente', keep='last')
      .copy()
)

import numpy as np

hoy = np.datetime64('today', 'D')

df_resumen['conteo_dia'] = np.busday_count(
    df_resumen['dia_ref'].values.astype('datetime64[D]'),
    hoy,
    weekmask='1111110'   # Lunes a sábado
)

# Crear env_1 hasta env_6
for i in range(1, 7):
    df_resumen[f'env_{i}'] = (df_resumen['q_envios'] == i).astype(int)

In [15]:
df_validar_pd_seleccion=df_validar_pd[['dni_cliente','COLOR_FINAL','USER_V3','FRESCURA','PROPENSION_DISTRIBUCION','OFERTA_MAX']]

In [16]:
df_resumen_1=df_resumen.merge(df_validar_pd_seleccion,on='dni_cliente',how='left')
df_resumen_1 = df_resumen_1.drop_duplicates(subset=['dni_cliente'])


In [17]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_resumen1.csv')
df_resumen_1.to_csv(ruta_archivo, index=False,sep=';')

In [54]:
fechas = pd.to_datetime(['2026-07-22'])

df_resumen_1_2=df_resumen_1[
    (df_resumen_1['fugas'] == 0) &
    (df_resumen_1['retiro'] == 0) &
    (df_resumen_1['dia_ref'].isin(fechas)) &
    (df_resumen_1['q_envios']!=1) &
    (df_resumen_1['PROPENSION_DISTRIBUCION'].notnull()) &
    (df_resumen_1['MONTO'].isna()) 
].copy()

In [55]:
df_resumen_1_2.shape

(1840, 23)

In [56]:
df_resumen_1_2['PROPENSION_DISTRIBUCION'].unique()

array(['4', '6', '2', '1', '5', '3'], dtype=object)

In [51]:
df_resumen_1_2.groupby('q_envios').size().reset_index(name='cantidad')

,q_envios,cantidad
0,2,2285


In [37]:
df_resumen_1.groupby(
    ['dia_ref', 'q_envios']
).size().reset_index(name='cantidad')

,dia_ref,q_envios,cantidad
0,2026-07-15,1,932
1,2026-07-15,2,458
2,2026-07-16,1,1708
3,2026-07-16,3,1


In [21]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [ ]:
df_resumen_1_2[df_resumen_1_2['MONTO'].isna()].head()

,dni_cliente,dia_ref,celular,target,MONTO,ASESOR,CANALVENTA,fugas,retiro,en_base,q_envios,conteo_dia,env_1,env_2,env_3,env_4,env_5,env_6,COLOR_FINAL,USER_V3,FRESCURA,PROPENSION_DISTRIBUCION,OFERTA_MAX
994,44355573,2026-07-09,910154097,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,NARANJA OSCURO,3. MES + PLD Peers,0,1,3000
995,44431542,2026-07-09,997580003,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,AMARILLO CLARO,2. sunedu & sunarp B,4,2,4200
996,16505328,2026-07-09,964869744,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,VERDE OSCURO,3. MES + PLD Peers,4,1,10600
997,15959521,2026-07-09,936093252,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,AMARILLO OSCURO,3. MES + PLD Peers,0,1,9800
998,15451499,2026-07-09,986015152,0,NaN,NaN,NaN,0,0,0,1,10,1,0,0,0,0,0,VERDE OSCURO,3. MES + PLD Peers,4,1,13400


In [65]:
df_resumen_1_2=df_resumen_1_2.rename(columns={'COLOR_FINAL':'color'})

In [66]:
df_resumen_1_2.shape

(1840, 23)

In [67]:
df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')


In [68]:
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')
df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1_2[['dni_cliente','color']],on='dni_cliente',how='inner')
df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1_2[['dni_cliente','color']],on='dni_cliente',how='inner')

In [69]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [70]:
df_prospectos_envio_alfin=df_prospectos_envio_alfin.drop(columns='color')

In [71]:
# df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')

df_prospectos_correos_alfin=df_prospectos_correos_alfin[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_prospectos_correos_alfin['tipo_carga']='MANUAL'

df_prospectos_envio_alfin=df_prospectos_envio_alfin[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()
df_prospectos_correos_alfin['fecha_visita']='2026-07-24'
df_prospectos_envio_alfin['fecha_visita']='2026-07-24'

df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset='dni_cliente')
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset='dni_cliente')

display(df_prospectos_correos_alfin.head(2))
display(df_prospectos_envio_alfin.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,21830475,ALFREDO ELEODORO REYES,AMARILLO OSCURO,7700.0,956559337,CHINCHA,2026-07-24,0 days 17:15:00,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,07122565,CARLOS CLIMACO MALLQUI BEJAR,AMARILLO OSCURO,9000.0,971146068,LOS OLIVOS,2026-07-24,0 days 10:15:00,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,16767136,FRANCISCO TIMANA BECERRA,954391558,737870 - VILLA MARIA 2,2026-07-24,7700,Derivacion
1,00000001,TARGET,47587005,LUIS ARNOLD YUPANQUI GUTIERREZ,963742019,734265 - TRUJ CENTRO,2026-07-24,14000,Derivacion


In [72]:
df_prospectos_correos_alfin.shape

(1840, 14)

In [73]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [74]:


df_prospectos_correos_alfin.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_prospectos_envio_alfin.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_14540\3762344665.py:1: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_prospectos_correos_alfin.to_sql(


1840

In [164]:
query = f"""
		SELECT dni_cliente FROM Alice.prospectos_envio_alfin 
    where fecha_visita='2026-07-20'
"""
df_estan = pd.read_sql(query, engine_mysql)

In [165]:
df_resumen_1 = df_resumen_1[
    ~df_resumen_1['dni_cliente'].isin(df_estan['dni_cliente'])
].copy()
df_resumen_1.shape

(0, 23)